[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/15_mlp_interview.ipynb)

# 🟠 Solution: SwiGLU MLP（面试版）

## 📋 题目描述

**难度：** Medium

实现 SwiGLU MLP（nn.Module）。

SwiGLU 是 LLaMA 等现代 LLM 使用的门控 MLP 变体：对一个投影应用 SiLU 门控，与另一个投影逐元素相乘。

**签名:** `SwiGLUMLP(d_model, d_ff)`（nn.Module）

**前向传播:** `forward(x) -> Tensor`
- `x` — 输入张量 (*, d_model)

**返回:** 输出张量 (*, d_model)

**约束:**
- 三个投影：gate_proj(d, d_ff)、up_proj(d, d_ff)、down_proj(d_ff, d)
- `forward(x) = down_proj(silu(gate_proj(x)) * up_proj(x))`
- SiLU(x) = x * sigmoid(x)

**提示：** `gate_proj(d→d_ff)`、`up_proj(d→d_ff)`、`down_proj(d_ff→d)`。前向：`down_proj(silu(gate_proj(x)) * up_proj(x))`。`silu(x) = x * sigmoid(x)`。


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ INTERVIEW

class SwiGLUMLP(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        # ---- Step 1: 定义三个线性投影 ----
        # SwiGLU 的"门控"机制：一个投影产生门控信号，另一个产生值
        # 相比标准 MLP（只有一个 up+down），SwiGLU 多了一个 gate_proj
        # 代价：参数量增加 50%（3个矩阵 vs 2个），但效果显著更好
        self.gate_proj = nn.Linear(d_model, d_ff)
        self.up_proj = nn.Linear(d_model, d_ff)
        self.down_proj = nn.Linear(d_ff, d_model)

    def forward(self, x):
        # ---- Step 2: 手写 SiLU 激活 ----
        # SiLU(x) = x * sigmoid(x)，面试中不能用 F.silu
        # sigmoid(x) = 1 / (1 + exp(-x))
        # SiLU 的特点：
        #   - x >> 0 时，sigmoid → 1，SiLU → x（近似恒等）
        #   - x << 0 时，sigmoid → 0，SiLU → 0（近似零）
        #   - x ≈ 0 时，SiLU 有平滑过渡，且有小的负值区域
        gate = self.gate_proj(x)  # (*, d_ff)
        silu_gate = gate * torch.sigmoid(gate)  # SiLU 手写版

        # ---- Step 3: 门控 × 值 → 降维 ----
        # up_proj(x) 是"候选值"，silu_gate 是"门控掩码"
        # 逐元素相乘：门控决定每个维度让多少信号通过
        up = self.up_proj(x)  # (*, d_ff)
        gated = silu_gate * up  # (*, d_ff)

        # down_proj: d_ff → d_model，回到原始维度
        return self.down_proj(gated)  # (*, d_model)

# 面试常见追问：
# Q: SwiGLU vs GELU MLP 的区别？
# A: SwiGLU 用门控机制（gate_proj × up_proj），GELU 版本只有一个 up_proj + 激活。
#    SwiGLU 参数多 50%，但 LLaMA/PaLM 等模型证明效果更好。
# Q: 为什么 SiLU 而不是 ReLU？
# A: SiLU 平滑且有负值区域，梯度更连续，训练更稳定。


In [ ]:
mlp = SwiGLUMLP(d_model=64, d_ff=128)
x = torch.randn(2, 8, 64)
print('Output:', mlp(x).shape)
print('Params:', sum(p.numel() for p in mlp.parameters()))


In [ ]:
from torch_judge import check
check('mlp')


## 📝 核心思路总结

1. **门控 MLP 结构**：gate_proj 生成门控信号，up_proj 生成候选值，逐元素相乘后 down_proj 降维
2. **SiLU 激活**：`x * sigmoid(x)`，平滑的 ReLU 替代品，零点附近有负值输出
3. **为什么 SwiGLU 更好**：门控机制让网络学习"选择性激活"，比单纯的非线性激活更灵活
4. **参数量权衡**：3 个投影 vs 标准 MLP 的 2 个，参数多 50%，但 LLaMA 等模型证明值得
